# 02 · 合成数据管线

HTML（`data-block-id`）→ Playwright 渲染 → DOM bbox → 同源 Markdown → 多点准星样本。

需要已执行：`uv sync --extra synth` 且 `playwright install chromium`。


In [ ]:
from pathlib import Path
import os, sys, json
ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

TEMPLATES = ROOT / "data" / "synth" / "templates"
OUT = ROOT / "data" / "processed" / "synth"
print("templates:", list(TEMPLATES.glob("*.html")))
print("out:", OUT)


In [ ]:
# 方式 A：直接跑仓库脚本（推荐）
!uv run python data/scripts/build_synth_batch.py --out data/processed/synth --noise --seed 0


In [ ]:
# 抽查：看 JSONL 条数与一条样本结构
from point_ocr.dataset_format import load_jsonl
from IPython.display import display
from PIL import Image

jsonl = OUT / "point_sharegpt.jsonl"
rows = load_jsonl(jsonl)
print("n_samples =", len(rows))
print(json.dumps(rows[0], ensure_ascii=False, indent=2)[:800])

img_path = rows[0]["images"][0]
display(Image.open(img_path).resize((480, 300)))
print("assistant target:", repr(rows[0]["messages"][1]["content"][:200]))


## 扩规模时你要做的事

1. 在 `data/synth/templates/` 增加更多带 `data-block-id` 的 HTML（双栏、表格、公式、代码、UI）
2. 质检：随机翻 marked 图，确认准星落在正确块、标签不是整页
3. 坏桶整类丢掉（见 `filter_qa.py`）
4. 目标：先 ~1k 手检，再 5万–20万对
